In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json

import ollama

from app.data_loader import DataLoader
from app.schemas import Question, Answer, GradingResult
from app.grading import grade_answer
from app.concepts_extraction import extract_key_concepts
from app.prompts import get_template
from app.schemas import ModelEvaluation
from app.utils import write_file

In [ ]:
test_cases = DataLoader("../data/generated_questions").test_cases

In [ ]:
models = {
    "qwen2-7b": "qwen2.5:7b",
    "qwen2-3b": "qwen2.5:3b",
    "qwen3-8b": "qwen3:8b",
    "tlite-8b": "t-tech/T-lite-it-2.1:Q4_K_M",
    "saiga-8b": "bambucha/saiga-llama3:8b",
    "vikhr-8b": "rscr/vikhr_llama3.1_8b:Q4_K_M",
    "llama3-8b": "llama3:8b-instruct-q4_K_M",
    "mistral-7b": "mistral:7b",
    "phi-8b": "phi3.5:3.8b-mini-instruct-q4_0",
    "yi-6b": "yi:6b",
}

In [ ]:
cases = test_cases[5:10]

In [ ]:
key_concepts = []

system_concept_template = get_template("concepts_extraction/system")
user_concept_template = get_template("concepts_extraction/user")

for case in cases:
    concepts, _ = extract_key_concepts(
        "qwen2.5:3b",
        0.9,
        system_concept_template.render(),
        user_concept_template.render({"question": case.question}),
    )
    key_concepts.append(concepts)

In [ ]:
key_concepts

In [ ]:
system_grading_template = get_template("grading/system")

In [ ]:
for temperature in [0.5, 0.7, 0.9, 1, 1.2, 1.5]:
    for model_short in models:
        model = models[model_short]

        print(f"{model_short} at t={temperature}")

        for i, test_case in enumerate(cases):
            concepts = key_concepts[i]

            evaluation = ModelEvaluation(
                model=model,
                temperature=temperature,
                test_case=test_case,
                key_concepts=concepts,
            )

            for j, answer in enumerate(test_case.answers):
                grading_result, response = grade_answer(
                    model,
                    temperature,
                    system_grading_template.render(
                        {"question": test_case.question, "key_concepts": key_concepts}
                    ),
                    answer.text,
                    len(concepts.concepts),
                )

                evaluation.grading_results.append(grading_result)
                if response.eval_duration and response.eval_count:
                    evaluation.times_per_token.append(
                        response.eval_duration // response.eval_count
                    )
                else:
                    evaluation.times_per_token.append(0)

                print(f"{i+1}/{len(cases)}, {j+1}/{len(test_case.answers)}")

            write_file(
                f"../output/{model_short}/{temperature}/{i}.json",
                evaluation.model_dump_json(),
            )